# 03 — SVD Model Training & Evaluation

Trains a baseline SVD collaborative filtering model on the processed Netflix dataset and evaluates it.

**Approach:** temporal train/test split — older ratings for training, newer ones for evaluation. This reflects real deployment conditions where the model must generalize to future interactions.

**Requires:** `02_preprocessing.ipynb` to have been run first.

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader, SVD, accuracy

### Parameters

In [24]:
ROOT                = Path('../')
PROCESSED_DATA_PATH = ROOT / 'data' / 'processed'

DATA_SAMPLE_FRACTION = 0.05   # fraction of processed data to train on
TEST_SIZE            = 0.2    # fraction held out for evaluation
TOP_N                = 10     # k for Precision@K and Recall@K
RELEVANCE_THRESHOLD  = 4.5    # minimum rating to consider a recommendation relevant

## 1. Load and sample data

Training on the full ~100M row dataset is too slow for local iteration. We take a **random** sample (`random_state=42` for reproducibility). Random sampling preserves the distribution of users, movies and ratings across the full catalog — slicing with `iloc[:N]` would only include the earliest movies and is heavily biased.

In [25]:
df = pd.read_parquet(PROCESSED_DATA_PATH / 'processed_data.parquet')
df = df.sample(frac=DATA_SAMPLE_FRACTION, random_state=42)

print(f"Sample: {len(df):,} rows ({DATA_SAMPLE_FRACTION:.0%} of full dataset)")
print(f"Unique users:  {df['customer_id'].nunique():,}")
print(f"Unique movies: {df['movie_id'].nunique():,}")

Sample: 5,019,717 rows (5% of full dataset)
Unique users:  424,565
Unique movies: 17,691


## 2. Temporal train/test split

We sort by date and split at the `1 - TEST_SIZE` quantile. All ratings **before** the split date are used for training; ratings **from** that date onward form the test set. This prevents data leakage — the model never sees future ratings during training.

In [26]:
sorted_idx = df['date'].argsort()
split_idx  = int(len(df) * (1 - TEST_SIZE))
split_date = df['date'].iloc[sorted_idx.iloc[split_idx]]

train_df = df[df['date'] < split_date]
test_df  = df[df['date'] >= split_date]

print(f"Split date: {split_date}")
print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")

Split date: 2005-08-08
Train: 4,010,496 rows | Test: 1,009,221 rows


## 3. Train SVD

We use Surprise's SVD implementation. The Surprise `Reader` defines the rating scale; `build_full_trainset()` converts the DataFrame into Surprise's internal format.

In [27]:
reader   = Reader(rating_scale=(1, 5))
trainset = Dataset.load_from_df(train_df[['customer_id', 'movie_id', 'rating']], reader).build_full_trainset()
testset  = list(zip(test_df['customer_id'], test_df['movie_id'], test_df['rating']))

model = SVD()
model.fit(trainset)
print("Training complete.")

Training complete.


## 4. Evaluate

- **RMSE:** measures average prediction error in rating units (1–5 scale).
- **Precision@K:** of the K recommendations shown to each user, what fraction would they actually rate highly (≥ threshold)?
- **Recall@K:** of all the movies a user would rate highly, what fraction appear in the top-K recommendations?

In [28]:
def precision_recall_at_k(predictions, k=10, threshold=4.0):
    user_ratings = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_ratings[uid].append((true_r, est))

    precisions, recalls = [], []
    for uid, ratings in user_ratings.items():
        top_k            = sorted(ratings, key=lambda x: x[1], reverse=True)[:k]
        n_relevant       = sum(r >= threshold for r, _ in ratings)
        n_relevant_top_k = sum(r >= threshold for r, _ in top_k)
        if n_relevant == 0:
            continue
        precisions.append(n_relevant_top_k / k)
        recalls.append(n_relevant_top_k / n_relevant)

    return np.mean(precisions), np.mean(recalls)

In [29]:
preds = model.test(testset)

rmse      = accuracy.rmse(preds, verbose=False)
precision, recall = precision_recall_at_k(preds, k=TOP_N, threshold=RELEVANCE_THRESHOLD)

print(f"RMSE:            {rmse:.4f}")
print(f"Precision@{TOP_N}:  {precision:.4f}")
print(f"Recall@{TOP_N}:     {recall:.4f}")

RMSE:            1.0253
Precision@10:  0.2067
Recall@10:     0.9576
